In [1]:
#| label: b0Fig3jn

# Prepare Python environment

import scipy.io as sio
from pathlib import Path

from contextlib import contextmanager
import sys, os
from pathlib import Path

@contextmanager
def suppress_stdout():
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:  
            yield
        finally:
            sys.stdout = old_stdout

import os
from pathlib import Path

def find_myst_yml_directories(start_dir=None):
    """
    Recursively search for directories containing myst.yml file.
    
    Args:
        start_dir (str or Path): Starting directory (defaults to current directory)
    
    Returns:
        list: List of full paths to directories containing myst.yml
    """
    if start_dir is None:
        start_dir = Path.cwd()
    else:
        start_dir = Path(start_dir)
    
    myst_dirs = []
    
    def _search_directory(current_dir):
        # Check if myst.yml exists in current directory
        myst_file = current_dir / "myst.yml"
        if myst_file.exists():
            myst_dirs.append(str(current_dir.resolve()))
            # Don't search subdirectories if we found myst.yml here
            return
        
        # Recursively search all subdirectories
        for item in current_dir.iterdir():
            if item.is_dir():
                try:
                    _search_directory(item)
                except (PermissionError, OSError):
                    # Skip directories we can't access
                    continue
    
    _search_directory(start_dir)
    return myst_dirs

def find_myst_yml_directories_upwards(start_dir=None):
    """
    Search for myst.yml in current directory, if not found go to parent and repeat.
    
    Args:
        start_dir (str or Path): Starting directory (defaults to current directory)
    
    Returns:
        str or None: Full path of directory containing myst.yml, or None if not found
    """
    if start_dir is None:
        current_dir = Path.cwd()
    else:
        current_dir = Path(start_dir)
    
    # Keep going up until we reach the filesystem root
    while current_dir != current_dir.parent:  # Stop at root
        myst_file = current_dir / "myst.yml"
        if myst_file.exists():
            return str(current_dir.resolve())
        
        # Move to parent directory
        current_dir = current_dir.parent
    
    return None

with suppress_stdout():
    repo_path = Path(find_myst_yml_directories_upwards())
    print(repo_path)
    data_req_path = repo_path / "binder" / "data_requirement.json"
    data_path = repo_path / "data"
    dataset_path = data_path / "qmrlab-mooc"
    
data_dir = dataset_path / "05-B0" / "05-B0" / "data"

import math
import json
import nibabel as nib
import numpy as np
from numpy.fft import ifftn, fftn, ifft, fftshift, ifftshift
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import butter, lfilter, freqz, filtfilt
from scipy.io import loadmat
import warnings
PI_UNICODE = "\U0001D70B"
CHI_UNICODE = "\U0001D712"
MICRO_UNICODE = "\u00B5"
GYRO_BAR_RATIO_H = 42.6e6  # [Hz/T]


# Load cylinder (Y 64)
susc = nib.load(data_dir / "field_simulations" / "cylinder" / "Chi.nii.gz").get_fdata()
fmap_hz_all = nib.load(data_dir / "field_simulations" / "cylinder" / "fmap_hz.nii.gz").get_fdata()
local_field_cyl = nib.load(data_dir  / "field_simulations" / "cylinder" / "local_field.nii.gz").get_fdata()

fig = make_subplots(rows=1, cols=3, shared_xaxes=False, horizontal_spacing=0.13, vertical_spacing = 0.12, subplot_titles=(f"Susceptibility distribution ({CHI_UNICODE})", "Simulated B0 map", "Simulated B0 map<br>no background field"), specs=[[{"type": "Heatmap"}, {"type": "Heatmap"}, {"type": "Heatmap"}]])
fig.add_trace(go.Heatmap(z=susc, colorscale='gray', colorbar_x=1/3 - 0.09, colorbar=dict(title="ppm", titleside="top")), 1, 1)
fig.add_trace(go.Heatmap(z=fmap_hz_all, colorscale='gray', colorbar_x=2/3 - 0.045, colorbar=dict(title="Hz", titleside="top")), 1, 2)
fig.add_trace(go.Heatmap(z=local_field_cyl, colorscale='gray', colorbar_x=1-0.004, colorbar=dict(title="Hz", titleside="top")), 1, 3)

### Create buttons for drop down menu
labels = ["Cylinders", "Brain"]
buttons = []
for i, label in enumerate(labels):
    if label == "Cylinders":
        visibility = [True, True, True, False, False, False]
    else:
        visibility = [False, False, False, True, True, True]
    button = dict(
                 label =  label,
                 method = 'update',
                 args = [{'visible': visibility},
                         {'title': " "}])
    buttons.append(button)

updatemenus = list([
    dict(active=0,
         x=0.58,
         y=1.5,
         buttons=buttons,
         showactive=True,
    )
])

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout({"height": 350, "width": 750},
                  title_text=" ",
                  title_x=0.5,
                  updatemenus=updatemenus,
                  showlegend=False
                 )
fig.show()

In [2]:
#| label: b0Fig3ajn

# Prepare Python environment

import scipy.io as sio
from pathlib import Path

import math
import json
import nibabel as nib
import numpy as np
from numpy.fft import ifftn, fftn, ifft, fftshift, ifftshift
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import butter, lfilter, freqz, filtfilt
from scipy.io import loadmat
import warnings
PI_UNICODE = "\U0001D70B"
CHI_UNICODE = "\U0001D712"
MICRO_UNICODE = "\u00B5"
GYRO_BAR_RATIO_H = 42.6e6  # [Hz/T]


# Load cylinder (Y 64)
susc = nib.load(data_dir / "field_simulations" / "cylinder" / "Chi.nii.gz").get_fdata()
fmap_hz_all = nib.load(data_dir / "field_simulations" / "cylinder" / "fmap_hz.nii.gz").get_fdata()
local_field_cyl = nib.load(data_dir  / "field_simulations" / "cylinder" / "local_field.nii.gz").get_fdata()

# Load brain (Z 210)
susc_brain = nib.load(data_dir / "field_simulations" / "brain" / "chi_masked.nii.gz").get_fdata()
fmap_hz_brain_all = nib.load(data_dir / "field_simulations" / "brain" / "fmap_masked.nii.gz").get_fdata()
local_field_brain = nib.load(data_dir / "field_simulations" / "brain" / "local_field.nii.gz").get_fdata()

# Rotate brain

susc_brain = np.rot90(susc_brain, -1)
fmap_hz_brain_all = np.rot90(fmap_hz_brain_all, -1)
local_field_brain = np.rot90(local_field_brain, -1)

# Remove the bulk offset: a brain in air is shifted by ~2000 Hz everywhere, which the scanner removes by setting its
# centre frequency. Show the field relative to the brain's median, empty outside the brain, on a range symmetric about 0.
brain_mask = fmap_hz_brain_all != 0
fmap_hz_brain_all = np.where(brain_mask, fmap_hz_brain_all - np.median(fmap_hz_brain_all[brain_mask]), np.nan)
b0_lim = float(np.percentile(np.abs(fmap_hz_brain_all[brain_mask]), 99))

fig = make_subplots(rows=1, cols=3, shared_xaxes=False, horizontal_spacing=0.13, vertical_spacing = 0.12, subplot_titles=(f"Susceptibility distribution ({CHI_UNICODE})", "Simulated B0 map", "Simulated B0 map<br>no background field"), specs=[[{"type": "Heatmap"}, {"type": "Heatmap"}, {"type": "Heatmap"}]])
fig.add_trace(go.Heatmap(z=susc, colorscale='gray', colorbar_x=1/3 - 0.09, colorbar=dict(title="ppm", titleside="top")), 1, 1)
fig.add_trace(go.Heatmap(z=fmap_hz_all, colorscale='gray', colorbar_x=2/3 - 0.045, colorbar=dict(title="Hz", titleside="top")), 1, 2)
fig.add_trace(go.Heatmap(z=local_field_cyl, colorscale='gray', colorbar_x=1-0.004, colorbar=dict(title="Hz", titleside="top")), 1, 3)
fig.add_trace(go.Heatmap(z=np.rot90(susc_brain, k=-1), colorscale='gray', colorbar_x=1/3 - 0.09, zmin=-0.5, zmax=0.5, colorbar=dict(title="ppm", titleside="top"), visible=False), 1, 1)
fig.add_trace(go.Heatmap(z=np.rot90(fmap_hz_brain_all, k=-1), colorscale='gray', colorbar_x=2/3 - 0.045, zmin=-b0_lim, zmax=b0_lim, colorbar=dict(title="Hz", titleside="top"), visible=False), 1, 2)
fig.add_trace(go.Heatmap(z=np.rot90(local_field_brain, k=-1), colorscale='gray', zmin=-4, zmax=4, colorbar_x=1-0.004, colorbar=dict(title="Hz", titleside="top"), visible=False), 1, 3)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
# No theme background or grid behind the empty outside of the brain
fig.update_layout(plot_bgcolor="rgba(0,0,0,0)")
fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=False, zeroline=False)
fig.update_layout({"height": 350, "width": 700},
                  title_text=" ",
                  title_x=0.5,
                  showlegend=False
                 )

# Add annotation for "a" (Figure 5-a)
fig.add_annotation(
    x=-0.07,  # X position (0 is left, 1 is right)
    y=-0.05,  # Y position (0 is bottom, 1 is top)
    xref="paper",  # Use paper coordinates (relative to the entire figure)
    yref="paper",
    text="a",  # Text to display
    showarrow=False,  # Don't show an arrow
    font=dict(size=50, color="black"),  # Font size and color
    xanchor="left",  # Anchor text to the left
    yanchor="bottom"  # Anchor text to the bottom
)


fig.show()

In [3]:
#| label: b0Fig3bjn

# Load brain (Z 210)
susc_brain = nib.load(data_dir / "field_simulations" / "brain" / "chi_masked.nii.gz").get_fdata()
fmap_hz_brain_all = nib.load(data_dir / "field_simulations" / "brain" / "fmap_masked.nii.gz").get_fdata()
local_field_brain = nib.load(data_dir / "field_simulations" / "brain" / "local_field.nii.gz").get_fdata()

# Rotate brain

susc_brain = np.rot90(susc_brain, -1)
fmap_hz_brain_all = np.rot90(fmap_hz_brain_all, -1)
local_field_brain = np.rot90(local_field_brain, -1)

# Remove the bulk offset: a brain in air is shifted by ~2000 Hz everywhere, which the scanner removes by setting its
# centre frequency. Show the field relative to the brain's median, empty outside the brain, on a range symmetric about 0.
brain_mask = fmap_hz_brain_all != 0
fmap_hz_brain_all = np.where(brain_mask, fmap_hz_brain_all - np.median(fmap_hz_brain_all[brain_mask]), np.nan)
b0_lim = float(np.percentile(np.abs(fmap_hz_brain_all[brain_mask]), 99))

fig_brain = make_subplots(rows=1, cols=3, shared_xaxes=False, horizontal_spacing=0.13, vertical_spacing = 0.12, subplot_titles=(f"Susceptibility distribution ({CHI_UNICODE})", "Simulated B0 map", "Simulated B0 map<br>no background field"), specs=[[{"type": "Heatmap"}, {"type": "Heatmap"}, {"type": "Heatmap"}]])

fig_brain.add_trace(go.Heatmap(z=np.rot90(susc_brain, k=-1), colorscale='gray', colorbar_x=1/3 - 0.09, zmin=-0.5, zmax=0.5, colorbar=dict(title="ppm", titleside="top")), 1, 1)
fig_brain.add_trace(go.Heatmap(z=np.rot90(fmap_hz_brain_all, k=-1), colorscale='gray', colorbar_x=2/3 - 0.045, zmin=-b0_lim, zmax=b0_lim, colorbar=dict(title="Hz", titleside="top")), 1, 2)
fig_brain.add_trace(go.Heatmap(z=np.rot90(local_field_brain, k=-1), colorscale='gray', zmin=-4, zmax=4, colorbar_x=1-0.004, colorbar=dict(title="Hz", titleside="top")), 1, 3)

fig_brain.update_xaxes(showticklabels=False)
fig_brain.update_yaxes(showticklabels=False)
# No theme background or grid behind the empty outside of the brain
fig_brain.update_layout(plot_bgcolor="rgba(0,0,0,0)")
fig_brain.update_xaxes(showgrid=False, zeroline=False)
fig_brain.update_yaxes(showgrid=False, zeroline=False)
fig_brain.update_layout({"height": 350, "width": 700},
                  title_text=" ",
                  title_x=0.5,
                  showlegend=False
                 )

# Add annotation for "a" (Figure 5-a)
fig_brain.add_annotation(
    x=-0.07,  # X position (0 is left, 1 is right)
    y=-0.05,  # Y position (0 is bottom, 1 is top)
    xref="paper",  # Use paper coordinates (relative to the entire figure)
    yref="paper",
    text="b",  # Text to display
    showarrow=False,  # Don't show an arrow
    font=dict(size=50, color="black"),  # Font size and color
    xanchor="left",  # Anchor text to the left
    yanchor="bottom"  # Anchor text to the bottom
)


fig_brain.show()

In [4]:
#| label: t2Fig7jn

import copy

# Create a deep copy of fig_brain
fig_inverse = copy.deepcopy(fig_brain)

# Remove the annotation ("b") from the new fig object
fig_inverse.layout.annotations = [ann for ann in fig.layout.annotations if ann.text != "b"]

# Flip the traces (susc_brain and local_field_brain) using data from fig_brain
# Swap z, zmin, and zmax between the first and third traces
(fig_inverse.data[0].z, fig_inverse.data[1].z) = (fig_brain.data[1].z, fig_brain.data[0].z)  # Swap z
(fig_inverse.data[0].zmin, fig_inverse.data[1].zmin) = (fig_brain.data[1].zmin, fig_brain.data[0].zmin)  # Swap zmin
(fig_inverse.data[0].zmax, fig_inverse.data[1].zmax) = (fig_brain.data[1].zmax, fig_brain.data[0].zmax)  # Swap zmax

# Swap the annotation titles by reusing the existing annotations
(fig_inverse.layout.annotations[0].text, fig_inverse.layout.annotations[1].text) = (
    fig_brain.layout.annotations[1].text, fig_brain.layout.annotations[0].text
)

# Remove the third trace
fig_inverse.data = fig_inverse.data[0:2]
fig_inverse.layout.annotations = fig_inverse.layout.annotations[0:2]

fig_inverse.update_layout({"height": 350, "width": 700},
                  title_text=" ",
                  title_x=0.5,
                  showlegend=False
                 )

                 

# Show the updated figure
fig_inverse.show()